# Assemble eligible tables

Builds the working datasets for the three London PICU sites (RBH, SMH, GOSH) from the raw supplier files.

**Input:** the raw site folders (read only) and the stay manifests in `derived/` (made in `manifests.ipynb`).

**What it does, for every table of a site:**
1. Keeps only rows that belong to a manifest stay (admitted 2013 or later, in PICANet, with real vitals).
2. Drops exact duplicate rows.
3. Writes one parquet file per table to `derived/assembled/<SITE>/`.

**What it does not do:** it drops no columns and changes no values. Every column is written as text, exactly as in the source. Anonymisation (new keys, relative times, free text, small cells) is a separate later step.

**Output:** row-level data. It stays in `derived/`, which is git-ignored, and never leaves the SDE. The logs printed here are counts only.

Run order: 1. `inventory_*` notebooks, 2. `manifests.ipynb`, 3. this notebook.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))

import inventory as I
import manifest as M
import assemble as A

smh_dir = Path("/home/jormond/Documents/DS217/users/jormond/SMH")
rbh_dir = Path("/home/jormond/Documents/DS217/users/jormond/RBH")
gosh_dir = Path("/home/jormond/Documents/DS217/users/jormond/GOSH")

## Directories and manifests

Loads the three saved manifests. They hold one row per eligible stay and define what is kept:

- **RBH:** matched by `stay_key` (patient + admission time, with the NIC/NND prefix removed).
- **SMH:** matched by `encounterId`.
- **GOSH:** matched by `project_id` and time. A row belongs to the stay whose window (admission start to discharge) contains its timestamp. The supplier's `admission_identifier` is not used, because it was built from the archived PICANet file, whose times disagree with the main file's.

Everything is read as text so IDs are never altered (for example by dropping leading zeros).

In [ ]:
derived = open_picu_dir / "derived"
out_dir = derived / "assembled"
out_dir.mkdir(exist_ok=True)

rbh_m = pd.read_csv(derived / "RBH_manifest.csv", dtype=str)
smh_m = pd.read_csv(derived / "SMH_manifest.csv", dtype=str)
gosh_m = pd.read_csv(derived / "GOSH_manifest.csv", parse_dates=["adm_time", "disch_time"])

## Preview

Lists which tables will be assembled for each site (headers only, so it is quick), and which cannot be linked to a stay and will be skipped.

Defaults: RBH skips the old PICANet file; SMH skips the two medication files that `medications_v2` already combines; GOSH uses only the merged per-variable files (not raw `lab_components`, the archived PICANet file, the old parquet or the logs). The GOSH PICANet file (`eprweights`) is handled separately.

Check the list before the long runs below.

In [ ]:
for site, d in [("RBH", rbh_dir), ("SMH", smh_dir), ("GOSH", gosh_dir)]:
    print(site)
    A.list_tables(site, d)

## RBH

For each table: reads in chunks, converts `studyID` to the stay key, keeps rows of manifest stays, then removes duplicates. `studyID` is replaced by `stay_key`.

Two counts are reported separately in the log:
- **exact duplicates removed:** identical rows.
- **merged (prefix only):** rows that differ only in the NIC/NND prefix. The same stay can appear under both prefixes in different tables (about 2,500 stays in the sedation table), so these are copies of one record, not new information.

In [ ]:
rbh_log = A.assemble_site("RBH", rbh_dir, out_dir, rbh_m)
rbh_log

In [ ]:
rbh_summary = A.summarise_log(rbh_log)

## SMH

Keeps rows whose `encounterId` (`encounterID` in the medication files) is in the manifest, then removes exact duplicate rows.

Expect a lot of duplicates in the long-format tables (vent, vitals, airway, blood gas, labs): the inventory found 29 to 52% exact repeats there. Whether they are true repeats or distinct measurements with identical fields is still an open question, so this only removes rows that are identical in every column.

The vent and vitals tables have about 10M rows, so this is the slow one.

In [ ]:
smh_log = A.assemble_site("SMH", smh_dir, out_dir, smh_m)
smh_log

In [ ]:
smh_summary = A.summarise_log(smh_log)

## GOSH

GOSH has no stay key in its tables, so rows are attached to stays by patient and time:
- the patient comes from `project_id`;
- the stay is the one for that patient whose window (admission start, with no margin, to discharge) contains `observation_datetime`;
- rows outside every stay (other wards, theatres, readmission gaps) are dropped, and `stay_id` is added to the rest.

The `eprweights` PICANet table is written with all 28 columns plus `stay_id`.

Tables without `project_id` or `observation_datetime` are not assembled; they are listed at the end of the run for a decision.

This is the longest run, with tables of millions of rows. Reruns skip tables that already have an output file.

## Check the time columns of the balance and flowsheet tables

`Net_Bal`, `Net_Bal_24` and the ventilation-mode flowsheet rows each have `chartTime`, `start_datetime` and `end_datetime`. If the start and end columns are an admission window repeated on every row (as in the archived PICANet file), they must not be used to place rows in time.

This samples each table (the first 500,000 rows) and prints counts only: how many rows parse, the date range, how many patients have a single value in the column, and how often each pair of columns agree to the minute. "Patients with one distinct value" near 100% would mean an admission-level constant, not a row time.

In [ ]:
pick = r"caboodle_fluid_balance|Total_fluids"
A.list_tables("GOSH", gosh_dir, include=pick)
A.time_candidates("GOSH", gosh_dir, include=pick)

In [ ]:
for rel in ["merged_balance/Net_Bal.csv",
            "merged_balance/Net_Bal_24.csv"]:
    A.time_column_check(gosh_dir, rel, ["chartTime", "start_datetime", "end_datetime"])

In [ ]:
checks = {
    "merged_vent_data/vent_Ventilation_mode_flowsheet_rows.csv": ["chart_time", "start_datetime", "end_datetime"],
}
for rel, cols in checks.items():
    A.time_column_check(gosh_dir, rel, cols)

In [ ]:
time_cols = {
    "July2026extraction_by_alex/pAIC/merged_antibiotics/antibiotics.csv": "chartTime",
    "merged_balance/Net_Bal.csv": "chartTime",
    "merged_balance/Net_Bal_24.csv": "chartTime",
    "merged_balance/Total_fluids.csv": "chartTime",
    "merged_vent_data/vent_Ventilation_mode_flowsheet_rows.csv": "chart_time",
}
gosh_log = A.assemble_site("GOSH", gosh_dir, out_dir, gosh_m, time_cols=time_cols)
gosh_log

## Summarise the assembly logs

Prints totals for a site's log (rows read, rows in manifest stays, duplicates removed, rows written), lists the tables where duplicates were removed, and shows the tables that keep the smallest share of their rows. Counts only, so the output can be pasted and saved.

In [ ]:
gosh_summary = A.summarise_log(gosh_log)

In [ ]:
pat = "Endotracheal|Hb|Haemoglobin|neutrophils|Neutrophils|wbc|WBC"
gosh_log[gosh_log["table"].str.contains(pat)][["table", "rows read", "in manifest stays", "rows written"]]

## Save the logs

The logs hold counts only (rows read, rows in stays, duplicates, rows written), so they can go in `reports/`. They record how many rows each table lost at each step.

In [ ]:
reports = open_picu_dir / "reports"
reports.mkdir(exist_ok=True)
for name, log in [("RBH", rbh_log), ("SMH", smh_log), ("GOSH", gosh_log)]:
    log.to_csv(reports / f"{name}_assembly_log.csv", index=False)

## One row per stay? (demographics, outcomes, PICANet)

These tables should have one row per stay, but a stay can have several rows that are not exact duplicates (for example repeated demographics rows, or conflicting outcomes). This counts rows per stay for each table, and lists which columns take different values between the rows of the same stay. Counts only.

If the rows of a stay disagree on outcome or discharge status, we need a rule for choosing before anonymisation.

In [ ]:
for site, key, names in [
    ("RBH", "stay_key", ["ai_picu_demographics", "ai_picu_outcomes", "ai_picu_piccanet"]),
    ("SMH", "encounterId", ["ai_picu_demographics", "ai_picu_outcomes", "ai_picu_piccanet"]),
]:
    for n in names:
        A.per_stay_conflicts(out_dir / site / f"{n}.parquet", key)
        print()